In [1]:
# Ensure required packages are installed in the notebook environment
import os
import shutil
import ssl
from doctr.models import ocr_predictor
from doctr.io import DocumentFile
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- CONFIGURATION ---
DB_DIR = "./chroma_db" # may need to reconfigure if updates are applied to ./reference-health-files
PDF_DATA_DIR = "./reference-health-files" # all PDFs needed for sourcing will be stored here
IMAGE_PATHS = ["john_doe_patient_statement.jpg", "john_doe_eob.jpg"] # medical bills to analyze

In [3]:
# --- INITIALIZE MODELS ---
# Using all-MiniLM-L6-v2 for semantic vectors as per your notes
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# LLM
llm = Ollama(model="llama3.2", temperature=0, num_ctx=4096) # make sure to run ollama pull llama3.2

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10945.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/f4/gqrvf9kn21vfz20ybyccg4j80000gn/T/ipykernel_252/3484588961.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2", temperature=0, num_ctx=4096) # make sure to run ollama pull llama3.2


In [4]:
# creation of vector that stores comprehensive information based on a folder of PDFs

def build_knowledge_base():
    """Builds the vector database from PDFs in the reference folder."""
    if not os.path.exists(PDF_DATA_DIR):
        os.makedirs(PDF_DATA_DIR)
        print(f"Created {PDF_DATA_DIR}. Add your PDFs there and run again.")
        return None

    print(f"Loading PDFs from {PDF_DATA_DIR}...")
    loader = DirectoryLoader(PDF_DATA_DIR, glob="./*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    
    # Recursive chunking to keep semantic meaning intact
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    
    print(f"Vectorizing {len(chunks)} chunks into ChromaDB...")
    # Delete old DB if it exists to prevent 'readonly' errors
    if os.path.exists(DB_DIR):
        shutil.rmtree(DB_DIR)
        
    vector_db = Chroma.from_documents(
        documents=chunks, 
        embedding=embeddings, 
        persist_directory=DB_DIR
    )
    print("Knowledge base built successfully.")
    return vector_db

In [5]:
# this function will turn 2 JPGs into a structured text that is ready to be analyzed

def extract_text_from_images(image_list):
    """Processes multiple JPGs and merges their text."""
    combined_text = ""
    
    # Initialize the model once outside the loop for speed
    model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
    
    for idx, img_path in enumerate(image_list):
        if not os.path.exists(img_path):
            print(f"Skipping: {img_path} not found.")
            continue
            
        print(f"Performing OCR on Page {idx + 1}: {img_path}...")
        doc = DocumentFile.from_images([img_path])
        result = model(doc)
        
        # Add a marker so the LLM knows where pages start/end
        combined_text += f"\n--- START OF PAGE {idx + 1} ---\n"
        
        for page in result.pages:
            for block in page.blocks:
                for line in block.lines:
                    line_text = " ".join([word.value for word in line.words])
                    combined_text += line_text + "\n"
                    
    return combined_text

In [13]:
def analyze_bill_with_rag(bill_text, target_language=input_language):
    """Uses RAG to explain the OCR text using the PDF knowledge base."""
    vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    
    template = """
    ### SYSTEM INSTRUCTION ###
    You are a Medical Billing Specialist, but don't state this at the beginning of the response. 
    CRITICAL: You will perform this task in {language}.

    ### CONSTRAINTS ###
    1. DIRECT ADDRESS: Speak directly to the user (e.g., use "You" in English,"Usted" or "Tu" in Spanish, "Vous" in French). 
       Do NOT refer to the user as "el paciente" or "le patient" in the third person.
    2. NO SIGN-OFFS: Do NOT include greetings, signatures, or placeholders like "[Votre nom]" or "Cordialement". 
       Start the analysis immediately.
    3. NO CHATBOT BEHAVIOR: Do not say "Here is your report" or "I hope this helps." 
    3. Be as simple, but straight to the point as possible. Be polite as well.
    4. Do not end with a question. Do not prompt the user to ask more questions.


    TASK:
    Compare the Statement to the EOB belonging to the user. Use the provided context to analyze the bills.
    Be factual, but also speak with less medical jargon and easier words.
    
    Additionally, make a recommendation based on context and current state of the bill of what steps a patient should take
    for financing their bill or if they should consider specific insurance coverage options.
    
    If the info isn't in the context, say you aren't sure.
    
    IMPORTANT:
    Be as simple, but straight to the point as possible. Be polite as well.
    Remember that you are addressing the user DIRECTLY. Do not say "the patient" or don't say the patient's name when summarizing.

    ### RESEARCH CONTEXT ###
    {context}

    ### PATIENT BILL ###
    {question}

    ### ANALYSIS REPORT ###
    """
    PROMPT = PromptTemplate(
        template=template, 
        input_variables=["context", "question"],
        partial_variables={"language": target_language} 
    )
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        chain_type_kwargs={"prompt": PROMPT},
        input_key="question" # Tells the chain to look for 'question'
    )
    
    print("Generating analysis in {target_language}...")
    response = qa_chain.invoke({"question": bill_text})

    return response["result"]

In [15]:
# dictionary for language input
language_dict = {
    1:"English",
    2:"Spanish",
    3:"French"
}

In [19]:
# EXECUTION

if __name__ == "__main__":
    ask_for_language = input("Enter 1 for English; Presione 2 para español; Tapez 3 pour le français: ")
    
    user_language = language_dict.get(int(ask_for_language), "English")
    ssl._create_default_https_context = ssl._create_unverified_context

    if not os.path.exists(DB_DIR):
        build_knowledge_base()
    
    # Define your list of images
    bill_images = ["john_doe_patient_statement.jpg", "john_doe_eob.jpg"]
    
    # Step A: Get all text from all pages
    full_bill_content = extract_text_from_images(bill_images)
    
    if full_bill_content.strip():
        # Step B: Run one single analysis on the total content
        analysis = analyze_bill_with_rag(full_bill_content, user_language)
        
        print("\n" + "="*40)
        print(f"\n--- ANALYSIS REPORT ({user_language}) ---")
        print("="*40)
        print(analysis)
    else:
        print("No text was extracted. Please check your image paths.")

Performing OCR on Page 1: john_doe_patient_statement.jpg...
Performing OCR on Page 2: john_doe_eob.jpg...
Generating analysis in {target_language}...


--- ANALYSIS REPORT (Spanish) ---
Usted tiene una factura de la NewYork-Presbyterian Weill Cornell Medical Center con un total de $3,550.00 en cargos. El pago total es de $1,600.00 y hay ajustes por $1,550.00.

La explicación de beneficios (EOB) de Aetna muestra que el paciente tiene una responsabilidad del 10% ($400.00) después de la cobertura de la planilla. Esto significa que usted debe pagar esta cantidad adicional para cubrir los cargos no cubiertos por la aseguradora.

Usted puede pagar en línea en nyp.org/paymybill o llamar a Aetna para obtener más información sobre sus opciones de financiamiento y cobertura.
